# **Fase 4: Optimizacion de Hiperparametros**

Estudiante: Maria Camila Navarrete Pinzón

Código: 2294353

Fecha: 08 febrero, 2026

# Notebook 9: Optimizacion de Hiperparametros

**Objetivo**: Encontrar la mejor combinación de hiperparámetros

**Estrategias:**

- **Grid Search + CV**: Búsqueda exhaustiva con cross-validation
- **Train-Validation Split**: Alternativa más rápida (un solo split)
- 
**Actividades**
  
1. Implementar Grid Search exhaustivo
2. Implementar Train-Validation Split
3. Comparar ambas estrategias
4. Seleccionar el mejor modelo global


## 1. Configuración de SparkSession

Se crea una sesión de spark configurada para ejecutarse en modo local, asignando memoria al driver.

In [7]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.sql.functions import col
import time

spark = SparkSession.builder \
    .appName("SECOP_HyperparameterTuning") \
    .master("local[*]")\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


## 2. Carga de datos

In [8]:
df = spark.read.parquet("/opt/spark-data/processed/secop_ml_ready.parquet")
df = df.withColumnRenamed("valor_del_contrato_num", "label") \
       .withColumnRenamed("features_pca", "features") \
       .filter(col("label").isNotNull())

train, test = df.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train.count():,}")
print(f"Test: {test.count():,}")


Train: 41,925


Test: 10,323


### 2.1 Modelo base y evaluador 

In [9]:
lr = LinearRegression(featuresCol="features", labelCol="label", maxIter=100)
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")

## 3. Reto 1: Diseñar grid de hiperparametros (escala logaritmica)

**Objetivo**: Crear un grid amplio pero inteligente.

**Parámetros a explorar**:
- `regParam`: Fuerza de regularización [0.01, 0.1, 1.0]
- `elasticNetParam`: Tipo de regularización [0.0, 0.5, 1.0]
- `maxIter`: Iteraciones máximas [50, 100, 200]

**Pregunta de diseño**: ¿Por qué usamos escala logarítmica para regParam (0.01, 0.1, 1.0) en lugar de lineal (0.33, 0.66, 1.0)?

In [10]:
grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .addGrid(lr.maxIter, [50, 100, 200]) \
    .build()

print(f"Combinaciones totales: {len(grid)}")


Combinaciones totales: 27


**Pregunta:** ¿Cuántas combinaciones hay?

3 (regParam) × 3 (elasticNetParam) × 3 (maxIter) = 27

**Pregunta:** ¿Cuántos modelos se entrenan con K=3

27 × 3 = 81 modelos entrenados
?

## 4. Reto 2: Implementar Grid Search + Cross-Validation

**Objetivo**: Ejecutar búsqueda exhaustiva con K-Fold CV.

**Instrucciones**:
1. Configura CrossValidator con tu grid
2. Usa K=3 (balance entre robustez y velocidad)
3. Ejecuta el entrenamiento
4. Registra el tiempo de ejecución
5. Obtén el mejor modelo y su RMSE en test

**Pregunta**: ¿Por qué K=3 y no K=5 para este experimento?


In [11]:
cv_grid = CrossValidator(
    estimator=lr,
    estimatorParamMaps=grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

print("Entrenando Grid Search + CV...")
start_time = time.time()
cv_grid_model = cv_grid.fit(train)
grid_time = time.time() - start_time

print(f"Completado en {grid_time:.2f} segundos")


Entrenando Grid Search + CV...


Completado en 78.34 segundos


### 4.1 Evaluación del mejor modelo

In [12]:
best_grid_model = cv_grid_model.bestModel
predictions_grid = best_grid_model.transform(test)
rmse_grid = evaluator.evaluate(predictions_grid)

print("\nMejor modelo")
print(f"regParam: {best_grid_model.getRegParam()}")
print(f"elasticNetParam: {best_grid_model.getElasticNetParam()}")
print(f"maxIter: {best_grid_model.getMaxIter()}")
print(f"RMSE Test: ${rmse_grid:,.2f}")



Mejor modelo
regParam: 1.0
elasticNetParam: 1.0
maxIter: 50
RMSE Test: $5,200,424,546.28


**Pregunta**: ¿Por qué K=3 y no K=5 para este experimento?  

Porque este grid es grande (27 combinaciones).

K=3 → 81 modelos, es un buen balance entre robustez y costo computacional.

K=5 → 135 modelos


## 5. Reto 3: 	Implementar Train-Validation Split

**Objetivo**: Implementar una alternativa más rápida al CV completo.

**Concepto**: TrainValidationSplit divide los datos de entrenamiento en un único split train/validation (ej: 80/20), en lugar de K folds.
**Ventaja**: Solo entrena cada modelo 1 vez (vs K veces en CV)
**Desventaja**: Menos robusto (depende de un solo split)

**Instrucciones**:
1. Usa TrainValidationSplit con el mismo grid
2. Configura trainRatio=0.8
3. Ejecuta y registra el tiempo
4. Compara con Grid Search + CV

In [13]:
tvs = TrainValidationSplit(
    estimator=lr,
    estimatorParamMaps=grid,
    evaluator=evaluator,
    trainRatio=0.8,
    seed=42
)

print("Entrenando con Train-Validation Split...")
start_time = time.time()
tvs_model = tvs.fit(train)
tvs_time = time.time() - start_time

print(f"Completado en {tvs_time:.2f} segundos")


Entrenando con Train-Validation Split...


Completado en 20.40 segundos


### 5.1 Evaluación 

In [14]:
best_tvs_model = tvs_model.bestModel
predictions_tvs = best_tvs_model.transform(test)
rmse_tvs = evaluator.evaluate(predictions_tvs)

print("\nMejor modelo")
print(f"regParam: {best_tvs_model.getRegParam()}")
print(f"elasticNetParam: {best_tvs_model.getElasticNetParam()}")
print(f"maxIter: {best_tvs_model.getMaxIter()}")
print(f"RMSE Test: ${rmse_tvs:,.2f}")



Mejor modelo
regParam: 1.0
elasticNetParam: 1.0
maxIter: 50
RMSE Test: $5,200,424,546.28


## 6. Reto 4: 	Comparar ambas estrategias (rendimiento vs velocidad)

**Objetivo**: Analizar las diferencias entre Grid Search + CV y Train-Validation Split.

**Instrucciones**:
1. Compara RMSE de ambas estrategias
2. Compara tiempos de ejecución
3. ¿Eligieron los mismos hiperparámetros?
4. ¿Cuándo usarías cada estrategia?

In [16]:

print("Comparación de estrategias")

print(f"Grid Search + CV:")
print(f"  - Tiempo: {grid_time:.2f}s")
print(f"  - RMSE Test: ${rmse_grid:,.2f}")

print(f"\nTrain-Validation Split:")
print(f"  - Tiempo: {tvs_time:.2f}s")
print(f"  - RMSE Test: ${rmse_tvs:,.2f}")


Comparación de estrategias
Grid Search + CV:
  - Tiempo: 78.34s
  - RMSE Test: $5,200,424,546.28

Train-Validation Split:
  - Tiempo: 20.40s
  - RMSE Test: $5,200,424,546.28


- Más rápido: Train-Validation Split
- Más confiable: Grid Search + CV
- ¿Mismos hiperparámetros?: A veces sí, a veces no. TVS depende de un solo split → más variabilidad.

## 7. Reto 5: 	Seleccionar y guardar modelo final con hiperparametros

**Objetivo**: Guardar el mejor modelo global y sus hiperparámetros.

**Instrucciones**:
1. Selecciona el mejor modelo entre ambas estrategias
2. Guarda el modelo en disco
3. Guarda los hiperparámetros en un JSON

In [17]:
mejor_modelo = best_grid_model if rmse_grid < rmse_tvs else best_tvs_model
model_path = "/opt/spark-data/processed/tuned_model"
mejor_modelo.save(model_path)

print(f"Mejor modelo guardado en: {model_path}")


Mejor modelo guardado en: /opt/spark-data/processed/tuned_model


### 7.1 Guardar hiperparámetros

In [20]:
import json

hiperparametros_optimos = {
    "regParam": float(mejor_modelo.getRegParam()),
    "elasticNetParam": float(mejor_modelo.getElasticNetParam()),
    "maxIter": int(mejor_modelo.getMaxIter()),
    "rmse_test": float(min(rmse_grid, rmse_tvs)),
    "estrategia": "Grid Search + CV" if rmse_grid < rmse_tvs else "Train-Validation Split"
}

with open("/opt/spark-data/processed/hiperparametros_optimos.json", "w") as f:
    json.dump(hiperparametros_optimos, f, indent=2)


## 8. Reto Bonus 1: 	Refinar grid alrededor de mejores valores

**Objetivo**: Refinar la búsqueda alrededor de los mejores hiperparámetros.

**Concepto**: Una vez identificada la mejor zona, crea un grid más fino alrededor de esos valores.

**Ejemplo**: Si el mejor regParam fue 0.1, prueba [0.05, 0.08, 0.1, 0.12, 0.15]

**Instrucciones**:
1. Toma los mejores hiperparámetros encontrados
2. Crea un grid fino alrededor de esos valores
3. Ejecuta CV con el grid fino
4. ¿Mejora el RMSE?

In [21]:
fine_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.8, 0.9, 1.0, 1.1, 1.2]) \
    .addGrid(lr.elasticNetParam, [0.8, 0.9, 1.0]) \
    .addGrid(lr.maxIter, [100]) \
    .build()

print(f"Combinaciones en el grid fino: {len(fine_grid)}")


Combinaciones en el grid fino: 15


In [22]:
cv_fine = CrossValidator(
    estimator=lr,
    estimatorParamMaps=fine_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

print("Entrenando Grid Fino con Cross-Validation...")
start_time = time.time()
cv_fine_model = cv_fine.fit(train)
fine_time = time.time() - start_time

print(f"Grid fino completado en {fine_time:.2f} segundos")


Entrenando Grid Fino con Cross-Validation...


Grid fino completado en 157.90 segundos


In [23]:
best_fine_model = cv_fine_model.bestModel
predictions_fine = best_fine_model.transform(test)
rmse_fine = evaluator.evaluate(predictions_fine)

print("\nMejor modelo")
print(f"regParam: {best_fine_model.getRegParam()}")
print(f"elasticNetParam: {best_fine_model.getElasticNetParam()}")
print(f"maxIter: {best_fine_model.getMaxIter()}")
print(f"RMSE Test (Grid Fino): ${rmse_fine:,.2f}")



Mejor modelo
regParam: 1.2
elasticNetParam: 1.0
maxIter: 100
RMSE Test (Grid Fino): $5,200,424,545.99


En el grid fino se refinó la búsqueda alrededor de los mejores hiperparámetros encontrados previamente.
El RMSE obtenido fue ligeramente menor / similar al del grid original, lo cual indica que el modelo ya se encontraba cerca de su óptimo.
El grid fino permite pequeños ajustes con mayor precisión, pero con un costo computacional menor que repetir una búsqueda exhaustiva completa.

## 9. Preguntas de reflexión

**¿Cuándo usarías Grid Search vs Random Search?**

*Respuesta:* Grid cuando el espacio es pequeño y bien definido. Random cuando el espacio es grande.

**¿Por qué Train-Validation Split es más rápido que CV?**

*Respuesta:* Porque cada combinación se entrena una sola vez.

**¿Qué pasa si el grid es demasiado grande?**

*Respuesta: Costo computacional explosivo adicionalmentee los entrenamientos se vuelven inviables.*

**¿Cómo implementarías Random Search en Spark ML?**

*Respuesta: Generando combinaciones aleatorias manualmente y pasándolas a ParamGridBuilder.*

In [24]:

print("Resúmen optimización de parámetros")

print("Verifica que hayas completado:")
print("  [✓] Diseñado grid de hiperparámetros")
print("  [✓] Ejecutado Grid Search + CV")
print("  [✓] Ejecutado Train-Validation Split")
print("  [✓] Comparado ambas estrategias")
print("  [✓] Guardado mejor modelo y hiperparámetros")

Resúmen optimización de parámetros
Verifica que hayas completado:
  [✓] Diseñado grid de hiperparámetros
  [✓] Ejecutado Grid Search + CV
  [✓] Ejecutado Train-Validation Split
  [✓] Comparado ambas estrategias
  [✓] Guardado mejor modelo y hiperparámetros


In [ ]:
spark.stop()